# Validación y métrica de negocio: M5 Forecast Pipeline

Evalúa el modelo final de la sesión pasada (**Random Forest sin regularizar**) y el segundo mejor (**Red neuronal**) con validación cruzada, con dos esquemas:

- **Temporal:** Cada pliegue entrena con días anteriores y valida con los 4 días siguientes.
- **Al azar:** `KFold` con las filas barajadas.

La validación cruzada se hace **solo dentro del conjunto de entrenamiento**, la semana de prueba no se toca.

In [1]:
import os
import time
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from pipeline_proyecto import NaiveEstacional, cargar_datos, dividir, nueva_preparacion
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:

datos = cargar_datos()
X_ent, X_prueba, y_ent, y_prueba, division = dividir(datos)
fechas_ent = datos.loc[X_ent.index, "DATE"]

print(f"Entrenamiento (donde se hace la validacion cruzada): {division['entrenamiento_desde']} "
      f"a {division['entrenamiento_hasta']}, {len(X_ent):,} filas, {fechas_ent.nunique()} dias")
del datos, X_prueba, y_prueba       # liberar memoria

Entrenamiento (donde se hace la validacion cruzada): 2016-04-20 a 2016-05-02, 396,370 filas, 13 dias


## 1. Métricas

**WAPE** = Σ|error| / Σ unidades reales: la métrica que se venía usando. Trata igual sobreestimar y subestimar.

**Costo asimétrico (métrica de negocio):** El pronóstico alimenta el stock de seguridad, y los dos errores no cuestan lo mismo:

- **Subestimar:** Falta producto en la góndola, se pierde la venta y su margen, y el cliente puede irse a otra tienda.
- **Sobreestimar:** Sobra inventario, cuesta mantenerlo (capital inmovilizado, espacio) y, en comida, puede vencerse.

`costo relativo = Σ precio × (1,0 × faltante + 0,2 × sobrante) / Σ precio × unidades reales`

In [3]:
COSTO_FALTANTE = 1.0   # por unidad no cubierta, como fraccion del precio
COSTO_SOBRANTE = 0.2   # por unidad que sobra, como fraccion del precio


def wape(real, pred):
    real, pred = np.asarray(real, float), np.asarray(pred, float)
    return np.abs(real - pred).sum() / real.sum()


def costo_relativo(real, pred, precio):
    real, pred, precio = (np.asarray(v, float) for v in (real, pred, precio))
    faltante = np.clip(real - pred, 0, None)
    sobrante = np.clip(pred - real, 0, None)
    costo = (precio * (COSTO_FALTANTE * faltante + COSTO_SOBRANTE * sobrante)).sum()
    return costo / (precio * real).sum()


def sesgo(real, pred):
    real, pred = np.asarray(real, float), np.asarray(pred, float)
    return (pred.sum() - real.sum()) / real.sum()

## 2. Los dos esquemas de validación

- **Temporal:** `TimeSeriesSplit` sobre los 26 días de entrenamiento, con 4 pliegues de 4 días.
- **Al azar:** `KFold(n_splits=4, shuffle=True, random_state=42)` sobre las mismas filas. Mezcla días: entrena con el futuro de lo que valida.

In [4]:
N_PLIEGUES = 3
dias = np.sort(fechas_ent.unique())

esquemas = []   # (esquema, pliegue, posiciones de entrenamiento, posiciones de validacion)

tss = TimeSeriesSplit(n_splits=N_PLIEGUES, test_size=4)
for k, (d_ent, d_val) in enumerate(tss.split(dias), start=1):
    pos_ent = np.flatnonzero(fechas_ent.isin(dias[d_ent]).to_numpy())
    pos_val = np.flatnonzero(fechas_ent.isin(dias[d_val]).to_numpy())
    esquemas.append(("temporal", k, pos_ent, pos_val))

kf = KFold(n_splits=N_PLIEGUES, shuffle=True, random_state=42)
for k, (pos_ent, pos_val) in enumerate(kf.split(X_ent), start=1):
    esquemas.append(("al_azar", k, pos_ent, pos_val))

for esquema, k, pos_ent, pos_val in esquemas:
    f_val = fechas_ent.iloc[pos_val]
    print(f"{esquema:<9} pliegue {k}: entrena {len(pos_ent):>7,} filas | valida {len(pos_val):>7,} filas "
          f"({f_val.min().date()} a {f_val.max().date()})")

temporal  pliegue 1: entrena  30,490 filas | valida 121,960 filas (2016-04-21 a 2016-04-24)
temporal  pliegue 2: entrena 152,450 filas | valida 121,960 filas (2016-04-25 a 2016-04-28)
temporal  pliegue 3: entrena 274,410 filas | valida 121,960 filas (2016-04-29 a 2016-05-02)
al_azar   pliegue 1: entrena 264,246 filas | valida 132,124 filas (2016-04-20 a 2016-05-02)
al_azar   pliegue 2: entrena 264,247 filas | valida 132,123 filas (2016-04-20 a 2016-05-02)
al_azar   pliegue 3: entrena 264,247 filas | valida 132,123 filas (2016-04-20 a 2016-05-02)


## 3. Modelos

Las mismas configuraciones de la sesión pasada. El naive estacional ("igual que hace 7 días") se agrega solo como referencia: no se entrena, así que no tarda nada.

In [5]:
MODELOS = {
    "Random Forest sin regularizar (final)": lambda: RandomForestRegressor(
        n_estimators=50, max_samples=0.3, n_jobs=-1, random_state=42),
    "Red neuronal (segundo)": lambda: MLPRegressor(
        hidden_layer_sizes=(64, 32), batch_size=1024, random_state=42),
    "Naive estacional (referencia)": lambda: NaiveEstacional(),
}


def nuevo_pipeline(fabrica):
    modelo = fabrica()
    preparacion = "passthrough" if isinstance(modelo, NaiveEstacional) else nueva_preparacion()
    return Pipeline([("preparacion", preparacion), ("modelo", modelo)])

## 4. Validación cruzada

In [6]:
PARCIAL = "pliegues_parcial.csv"
filas = pd.read_csv(PARCIAL).to_dict("records") if os.path.exists(PARCIAL) else []
hechos = {(f["esquema"], f["pliegue"], f["modelo"]) for f in filas}

for esquema, k, pos_ent, pos_val in esquemas:
    for nombre, fabrica in MODELOS.items():
        if (esquema, k, nombre) in hechos:
            continue                                  # ya calculado en una corrida anterior
        X_a, y_a = X_ent.iloc[pos_ent], y_ent.iloc[pos_ent]
        X_v, y_v = X_ent.iloc[pos_val], y_ent.iloc[pos_val]
        f_val = fechas_ent.iloc[pos_val]

        pipe = nuevo_pipeline(fabrica)
        inicio = time.perf_counter()
        pipe.fit(X_a, y_a)
        segundos = time.perf_counter() - inicio
        pred = pipe.predict(X_v)
        filas.append({
            "modelo": nombre, "esquema": esquema, "pliegue": k,
            "valida_desde": str(f_val.min().date()), "valida_hasta": str(f_val.max().date()),
            "filas_entrenamiento": len(pos_ent), "filas_validacion": len(pos_val),
            "wape": wape(y_v, pred),
            "costo_relativo": costo_relativo(y_v, pred, X_v["SELL_PRICE"]),
            "sesgo": sesgo(y_v, pred),
            "segundos_entrenamiento": segundos,
        })
        print(f"{esquema:<9} pliegue {k} | {nombre:<38} WAPE {filas[-1]['wape']:.4f} | "
              f"costo {filas[-1]['costo_relativo']:.4f} | sesgo {filas[-1]['sesgo']:+.3f} | {segundos:.0f} s")

        pd.DataFrame(filas).to_csv(PARCIAL, index=False)   # guarda el avance
        del pipe, pred, X_a, y_a, X_v, y_v                  # libera memoria
        gc.collect()

pliegues = pd.DataFrame(filas)

temporal  pliegue 1 | Random Forest sin regularizar (final)  WAPE 0.8214 | costo 0.5635 | sesgo -0.063 | 1 s
temporal  pliegue 1 | Red neuronal (segundo)                 WAPE 0.9927 | costo 0.5468 | sesgo +0.272 | 173 s
temporal  pliegue 1 | Naive estacional (referencia)          WAPE 0.8593 | costo 0.5428 | sesgo +0.029 | 0 s
temporal  pliegue 2 | Random Forest sin regularizar (final)  WAPE 0.8893 | costo 0.5821 | sesgo -0.010 | 4 s
temporal  pliegue 2 | Red neuronal (segundo)                 WAPE 1.0806 | costo 0.5664 | sesgo +0.333 | 671 s
temporal  pliegue 2 | Naive estacional (referencia)          WAPE 0.9317 | costo 0.5882 | sesgo +0.018 | 0 s
temporal  pliegue 3 | Random Forest sin regularizar (final)  WAPE 0.8072 | costo 0.5449 | sesgo -0.062 | 11 s
temporal  pliegue 3 | Red neuronal (segundo)                 WAPE 1.0568 | costo 0.5397 | sesgo +0.298 | 1149 s
temporal  pliegue 3 | Naive estacional (referencia)          WAPE 0.8178 | costo 0.5494 | sesgo -0.061 | 0 s
al_azar   p

## 5. Tabla: cada pliegue, promedio y desviación

In [7]:
METRICAS = ["wape", "costo_relativo", "sesgo", "segundos_entrenamiento"]

resumen = (pliegues.groupby(["modelo", "esquema"])[METRICAS]
           .agg(["mean", "std"]))
resumen.columns = [f"{m}_{e}" for m, e in resumen.columns]

promedio = pliegues.groupby(["modelo", "esquema"], as_index=False)[METRICAS].mean().assign(pliegue="promedio")
desviacion = pliegues.groupby(["modelo", "esquema"], as_index=False)[METRICAS].std().assign(pliegue="desviacion")
tabla = pd.concat([pliegues, promedio, desviacion], ignore_index=True)
tabla["pliegue"] = tabla["pliegue"].astype(str)
tabla = tabla.sort_values(["esquema", "modelo"], kind="stable")
tabla.to_csv("validacion_proyecto.csv", index=False)
print("Guardado validacion_proyecto.csv")

resumen.round(4)

Guardado validacion_proyecto.csv


wape_mean  wape_std  \
modelo                                esquema                         
Naive estacional (referencia)         al_azar      0.8684    0.0027   
                                      temporal     0.8696    0.0576   
Random Forest sin regularizar (final) al_azar      0.8515    0.0013   
                                      temporal     0.8393    0.0439   
Red neuronal (segundo)                al_azar      0.8495    0.0072   
                                      temporal     1.0434    0.0455   

                                                costo_relativo_mean  \
modelo                                esquema                         
Naive estacional (referencia)         al_azar                0.5608   
                                      temporal               0.5601   
Random Forest sin regularizar (final) al_azar                0.5427   
                                      temporal               0.5635   
Red neuronal (segundo)                al_azar                0.5643   
                                      temporal               0.5510   

                                                costo_relativo_std  \
modelo                                esquema                        
Naive estacional (referencia)         al_azar               0.0044   
                                      temporal              0.0246   
Random Forest sin regularizar (final) al_azar               0.0029   
                                      temporal              0.0186   
Red neuronal (segundo)                al_azar               0.0059   
                                      temporal              0.0138   

                                                sesgo_mean  sesgo_std  \
modelo                                esquema                           
Naive estacional (referencia)         al_azar      -0.0075     0.0003   
                                      temporal     -0.0047     0.0487   
Random Forest sin regularizar (final) al_azar       0.0270     0.0033   
                                      temporal     -0.0450     0.0300   
Red neuronal (segundo)                al_azar       0.0092     0.0243   
                                      temporal      0.3011     0.0306   

                                                segundos_entrenamiento_mean  \
modelo                                esquema                                 
Naive estacional (referencia)         al_azar                        0.0003   
                                      temporal                       0.0005   
Random Forest sin regularizar (final) al_azar                       11.1947   
                                      temporal                       5.2183   
Red neuronal (segundo)                al_azar                      935.5311   
                                      temporal                     664.3231   

                                                segundos_entrenamiento_std  
modelo                                esquema                               
Naive estacional (referencia)         al_azar                       0.0000  
                                      temporal                      0.0002  
Random Forest sin regularizar (final) al_azar                       0.8133  
                                      temporal                      5.1318  
Red neuronal (segundo)                al_azar                     217.5112  
                                      temporal                    488.0504

## 6. Comparaciones para las respuestas

In [8]:
def fila(modelo, esquema):
    return resumen.loc[(modelo, esquema)]

FINAL, SEGUNDO = "Random Forest sin regularizar (final)", "Red neuronal (segundo)"

for esquema in ["temporal", "al_azar"]:
    f, s = fila(FINAL, esquema), fila(SEGUNDO, esquema)
    print(f"\n== Esquema {esquema} ==")
    for metrica in ["wape", "costo_relativo"]:
        dif = s[f"{metrica}_mean"] - f[f"{metrica}_mean"]
        print(f"{metrica:<15} final {f[f'{metrica}_mean']:.4f} ± {f[f'{metrica}_std']:.4f} | "
              f"segundo {s[f'{metrica}_mean']:.4f} ± {s[f'{metrica}_std']:.4f} | "
              f"diferencia (segundo - final) {dif:+.4f}")

# Diferencia pliegue a pliegue: los dos modelos se evaluan en los mismos pliegues,
# asi que se puede ver si el final gana en todos o solo en promedio.
for esquema in ["temporal", "al_azar"]:
    p = pliegues[pliegues["esquema"] == esquema].pivot(index="pliegue", columns="modelo", values="wape")
    gana = (p[FINAL] < p[SEGUNDO]).sum()
    print(f"\n{esquema}: el final tiene menor WAPE en {gana} de {len(p)} pliegues")
    print((p[SEGUNDO] - p[FINAL]).round(4).rename("WAPE segundo - final").to_string())


== Esquema temporal ==
wape            final 0.8393 ± 0.0439 | segundo 1.0434 ± 0.0455 | diferencia (segundo - final) +0.2041
costo_relativo  final 0.5635 ± 0.0186 | segundo 0.5510 ± 0.0138 | diferencia (segundo - final) -0.0125

== Esquema al_azar ==
wape            final 0.8515 ± 0.0013 | segundo 0.8495 ± 0.0072 | diferencia (segundo - final) -0.0020
costo_relativo  final 0.5427 ± 0.0029 | segundo 0.5643 ± 0.0059 | diferencia (segundo - final) +0.0215

temporal: el final tiene menor WAPE en 3 de 3 pliegues
pliegue
1    0.1713
2    0.1914
3    0.2496

al_azar: el final tiene menor WAPE en 1 de 3 pliegues
pliegue
1   -0.0077
2   -0.0026
3    0.0041
